# 00_bootstrap (Colab temporal)

Ejecutá **Run all**. Este notebook deja listo el repo para trabajar temporalmente en Colab sin tocar archivos fuera de `colab/`.

In [ ]:
# Parámetros de arranque (editables)
REPO_URL = ""  # Ejemplo: https://github.com/<org>/<repo>.git
REPO_DIR = "/content/geostats-colab-lab"
MOUNT_DRIVE = False

# Si querés forzar persistencia en Drive, luego podés mover outputs allí.


In [ ]:
import platform
import sys

print("Python version:", platform.python_version())
print("Executable:", sys.executable)


In [ ]:
# (Opcional) montar Google Drive
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive montado en /content/drive')
else:
    print('Drive no montado (MOUNT_DRIVE=False).')


In [ ]:
# Clonar o actualizar repositorio
import os
import subprocess

if os.path.isdir(REPO_DIR) and os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print(f'Repo ya existe en {REPO_DIR}. Haciendo git pull...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    if not REPO_URL:
        raise RuntimeError('Definí REPO_URL para clonar el repo en /content.')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

print('Repo path:', REPO_DIR)


In [ ]:
# Instalar dependencias mínimas para flujo moderno
import subprocess
import sys

req_path = f"{REPO_DIR}/colab/requirements_colab.txt"
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', req_path], check=True)
print('Requirements instalados desde', req_path)


In [ ]:
# Preparar bootstrap helper
import os
import sys
from pathlib import Path

repo_root = Path(REPO_DIR).resolve()
os.chdir(repo_root)

bootstrap_path = repo_root / 'colab'
if str(bootstrap_path) not in sys.path:
    sys.path.insert(0, str(bootstrap_path))

import bootstrap

print('CWD:', Path.cwd())
print('Repo root:', repo_root)


In [ ]:
# Forzar import desde src/ y validar ruta efectiva importada
from pathlib import Path

repo_root = Path(REPO_DIR).resolve()
module_file = bootstrap.ensure_src_import(repo_root)

print('Ruta efectiva de geostats_pipeline importado:')
print(module_file)
assert '/src/geostats_pipeline' in str(module_file).replace('\\', '/'), 'Import no resuelto desde src/'
print('OK: import resuelto desde src/geostats_pipeline')


In [ ]:
# Cargar config temporal y materializar runtime config compatible con Colab
from pathlib import Path

repo_root = Path(REPO_DIR).resolve()
runtime_cfg = bootstrap.materialize_runtime_config(repo_root)
raw_cfg = bootstrap.load_yaml(repo_root / 'colab' / 'config.colab.yml')

print('Config base:', repo_root / 'colab' / 'config.colab.yml')
print('Runtime config:', runtime_cfg)
print('data.path (base):', raw_cfg['data']['path'])
print('outputs.base_dir (base):', raw_cfg['outputs']['base_dir'])


In [ ]:
# Smoke test real: stage=setup
run_dir = bootstrap.smoke_test_setup(runtime_cfg)
print('Smoke test OK. Run generado en:')
print(run_dir)


In [ ]:
# Listo para trabajar: helper manual para correr otras etapas
from geostats_pipeline.steps import run_pipeline

def run_stage(stage: str):
    return run_pipeline(str(runtime_cfg), stage=stage)

print('Entorno listo. Ejemplos:')
print("run_stage('qaqc')")
print("run_stage('variography')")
print("run_stage('kriging')")
print("run_stage('validation')")
